In [1]:
%load_ext autoreload
%autoreload 2
import dask
import dask.distributed
from dask_util import DaskClient
import dask_util
import numpy as np

In [2]:
local_params = {
    "n_workers":4, 
    "processes" : True, 
    "dashboard_address" : 'localhost:7777'
    
}

# cluster = {
#     "cores" : 24,
#     "processes" : 1,
#     "memory" : "1GB",
#     "shebang" : '#!/usr/bin/env bash',
#     "queue" : "serc",
#     "walltime" : "00:10:00",
#     "local_directory" : '/tmp',
#     "death_timeout" : "15s",
#     "interface" : "ib0",
#     "log_directory" : f'{os.environ["SCRATCH"]}/dask_jobqueue_logs/'    
# }


client = DaskClient(local_params=local_params)

2023-03-07 22:37:48,717 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-dkl6u3bj', purging
2023-03-07 22:37:48,717 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-jgacq4zd', purging


In [3]:
%load_ext autoreload
%autoreload 2
import SepVector
from __pyDaskVector import DaskVector
from __pyDaskOperator import DaskOperator
import Hypercube
import pyOperator as Op


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
WARNING! DATAPATH not found. The folder /tmp will be used to write binary files


/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


In [53]:

ns = [10,4]
os = [0,0]
ds = [1,1]
chunks = (1,3)

ax = Hypercube.axis(n=1, o=1, d=1)
hyp = Hypercube.hypercube(ns=ns, ds=ds, os=os)
vec = SepVector.getSepVector(ns=ns, ds=ds, os=os)
vec.set(1)

floatVector
Axis 1: n=10	o=0.000000	d=1.000000
Axis 2: n=4	o=0.000000	d=1.000000

In [54]:
# option 1
# creating from scratch
data = DaskVector(client, vecCls=SepVector.floatVector, ns=ns, ds=ds, os=os, chunks=chunks)

In [55]:
data[:]

[array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)]

In [60]:
data.writeVec('test.H')

In [7]:
# option 2
# creating from existing in-memory SepVector
daskVec = DaskVector(client, from_vector=vec, chunks=chunks)

In [8]:
client.getClient().has_what()

{'tcp://127.0.0.1:34037': ('floatVector-ffe1f679b4148bb63385bc7a987a08bf',
  'window-a1e9b0e047b708dba67302d2d5816e4a'),
 'tcp://127.0.0.1:34881': ('floatVector-2f359ec81387ecec692bbd5f2f721903',
  'window-a2c9eeda7e12ad152af7135ed2d0a5a4'),
 'tcp://127.0.0.1:35013': ('floatVector-d58465734b5965bd2fc523b5102931fe',),
 'tcp://127.0.0.1:41503': ('window-0a14ea998683eb643aa247bef7eb40d1',)}

In [9]:
daskVec.fut

[<Future: finished, type: SepVector.floatVector, key: window-0a14ea998683eb643aa247bef7eb40d1>,
 <Future: finished, type: SepVector.floatVector, key: window-a2c9eeda7e12ad152af7135ed2d0a5a4>,
 <Future: finished, type: SepVector.floatVector, key: window-a1e9b0e047b708dba67302d2d5816e4a>]

In [10]:
scaleOp = DaskOperator(client, opCls=Op.scalingOp, domain=daskVec, range=data, op_args=[4])

In [11]:
data[:]

[array([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]], dtype=float32),
 array([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]], dtype=float32)]

In [46]:
scaleOp.forward(False, daskVec, data)

In [47]:
data[:]

[array([[64., 64., 64.],
        [64., 64., 64.],
        [64., 64., 64.],
        [64., 64., 64.]], dtype=float32),
 array([[64., 64., 64.],
        [64., 64., 64.],
        [64., 64., 64.],
        [64., 64., 64.]], dtype=float32),
 array([[64., 64., 64., 64.],
        [64., 64., 64., 64.],
        [64., 64., 64., 64.],
        [64., 64., 64., 64.]], dtype=float32)]

In [42]:
scaleOp.adjoint(False, daskVec, data)

In [43]:
daskVec[:]

[array([[16., 16., 16.],
        [16., 16., 16.],
        [16., 16., 16.],
        [16., 16., 16.]], dtype=float32),
 array([[16., 16., 16.],
        [16., 16., 16.],
        [16., 16., 16.],
        [16., 16., 16.]], dtype=float32),
 array([[16., 16., 16., 16.],
        [16., 16., 16., 16.],
        [16., 16., 16., 16.],
        [16., 16., 16., 16.]], dtype=float32)]